# Day 20: Abstraction & Abstract Classes
## Hide the "How", Show Only the "What"

---

### Today's Roadmap

| Part | Topic | The Question It Answers |
|------|-------|-------------------------|
| **0** | Quick Recap | What do we already know? |
| **1** | What is Abstraction? | What does it mean in real life? |
| **2** | You Already Use It! | Where have I seen abstraction before? |
| **3** | The Problem with Normal Parents | Why isn't inheritance alone enough? |
| **4** | Abstract Classes Step by Step | How do I use `ABC` and `@abstractmethod`? |
| **5** | Abstract + Normal Methods Together | Can an abstract class have real code? |
| **6** | Real Example: Payment System | How is this used in real apps? |
| **7** | Abstraction vs Encapsulation | What's the difference? |
| **8** | Mini Project: Library System | Everything from Days 16–20 together |
| **9** | Common Mistakes | What usually goes wrong? |
| **10** | Practice Exercises + Solutions | Can I do it myself? |
| **11** | Cheat Sheet & The 4 Pillars of OOP | What do I need to remember? |

> **How to use this notebook:** Read the explanation, **guess the output**, then run the cell.

---

## Part 0: Quick Recap of Day 19

- **Inheritance:** `class Dog(Animal):` → the child gets everything from the parent
- **`super()`** → call the parent's version of a method
- **Overriding** → the child writes its own version of a parent's method
- **Polymorphism** → the same call (`animal.speak()`) behaves differently for each object

Today's lesson builds **directly** on inheritance and overriding.

---

## Part 1: What is Abstraction?

> **Abstraction = hiding the complicated "HOW", and showing only a simple "WHAT".**

The user only needs to know **what** something does — not **how** it does it inside.

### Real-life examples

| Thing | What you see (simple) | What's hidden (complex) |
|-------|------------------------|-------------------------|
| 🚗 **Car** | Steering, accelerator, brake | Engine, fuel injection, gears |
| ☕ **Coffee machine** | One "Espresso" button | Heating water, grinding beans, pressure |
| 📺 **TV remote** | Volume + / − buttons | Infrared signals, circuit boards |
| 📱 **WhatsApp** | Type a message, press Send | Encryption, servers, internet protocols |

```
┌─────────────────────────────────────────┐
│         WHAT THE USER SEES              │
│                                         │
│      [ Send ]  ──►  "Message sent ✓"    │
├─────────────────────────────────────────┤
│         WHAT IS HIDDEN                  │
│                                         │
│   • connect to the internet             │
│   • encrypt the message                 │
│   • find the receiver's phone           │
│   • retry if the network fails          │
└─────────────────────────────────────────┘
```

**Why it matters:** Abstraction lets us use (and build) big programs **without having to think about every tiny detail at once**.

---

## Part 2: You Already Use Abstraction Every Day!

Every built-in function hides a lot of work behind a simple name.

In [ ]:
numbers = [5, 2, 9, 1, 7]

numbers.sort()          # You don't know HOW it sorts (a clever algorithm called Timsort)
print(numbers)          # ...and you don't need to!

print(len(numbers))     # You don't count the items yourself
print(max(numbers))     # You don't write a loop to find the biggest

### Functions are abstraction too

Every time you write a **function**, you create an abstraction: the caller only needs to know the **name** and the **inputs** — not the formula inside.

In [ ]:
# The formula is HIDDEN inside the function
def final_price(price, discount=0, tax=0.18):
    after_discount = price * (1 - discount)
    return round(after_discount * (1 + tax), 2)


# The caller only cares about WHAT it does, not HOW
print('Laptop :', final_price(80000, discount=0.10))
print('Mouse  :', final_price(1500, discount=0.05))
print('Monitor:', final_price(25000))

✅ If the tax rule changes tomorrow, we fix it in **one place** — the callers don't change at all.

Today we'll learn the OOP way of doing abstraction: **abstract classes**.

---

## Part 3: The Problem with Normal Parent Classes

Let's make a `Shape` parent. **Every** shape must have an `area()` method, so each child is supposed to override it.

But what if a programmer **forgets**?

In [ ]:
class Shape:
    def area(self):
        return 0                         # "Children, please override this!"


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):                      # ✅ Remembered to override
        return self.side ** 2


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    # ❌ FORGOT to write area()!


shapes = [Square(4), Circle(3)]
for s in shapes:
    print(type(s).__name__, 'area =', s.area())

😱 **The circle's area is `0`, and there's no error!** The bug is silent — maybe nobody notices until a customer gets a wrong bill.

There's a second problem too:

In [ ]:
s = Shape()             # What shape is this? A "shape" with no sides?
print(s.area())         # It makes no sense, but Python allows it

### What we really want

1. ❌ Nobody should be able to create a plain `Shape()` — it's just an **idea**, not a real shape.
2. ✅ Every child **must** write `area()` — and Python should **complain immediately** if it's missing.

This is exactly what an **abstract class** does. 👇

---

## Part 4: Abstract Classes — Step by Step

An **abstract class** is a class that:
- is only a **blueprint / contract** — you **cannot** create objects from it
- lists methods that **every child MUST write**

**Analogy — a job contract 📝:** The contract says *"To work here, you MUST be able to do X and Y."* The contract itself doesn't do any work. The **employees** who sign it do the actual work, each in their own way.

### The 3 tools you need

| Tool | What it does |
|------|--------------|
| `from abc import ABC, abstractmethod` | Import the tools (`abc` = **A**bstract **B**ase **C**lasses) |
| `class Shape(ABC):` | Makes `Shape` an abstract class |
| `@abstractmethod` | Marks a method as **"every child MUST write this"** |

### Step 1 — Create the abstract class (the contract)

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):                        # 1) Inherit from ABC

    @abstractmethod                      # 2) Mark the required method
    def area(self):
        pass                             # No code here - children must write it

### Step 2 — Try to create a `Shape` directly

In [ ]:
try:
    s = Shape()
except TypeError as e:
    print('Error:', e)

✅ **Blocked!** A `Shape` is just an idea, so Python won't let you create one.

### Step 3 — A child that follows the contract ✅

In [ ]:
class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):                      # Contract fulfilled
        return self.side ** 2


sq = Square(4)                           # Works!
print('Square area =', sq.area())

### Step 4 — A child that breaks the contract ❌

Now let's "forget" `area()` again, just like in Part 3:

In [ ]:
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    # ❌ FORGOT to write area()


try:
    c = Circle(3)
except TypeError as e:
    print('Error:', e)

🎉 **This time Python catches the mistake immediately** — the moment we try to create the object — instead of silently giving a wrong answer.

This idea is called **"fail fast"**: find the bug **early**, while you are writing the code, not later when a real user is affected.

```
 Normal parent class               Abstract class
 ───────────────────               ──────────────
 forget area()                     forget area()
       │                                 │
       ▼                                 ▼
 object is created                 ❌ TypeError right away
       │                              "fix me before you run!"
       ▼
 wrong answer (0) later 😱
```

Let's fix `Circle` properly:

In [ ]:
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):                      # ✅ Contract fulfilled
        return round(3.14159 * self.radius ** 2, 2)


shapes = [Square(4), Circle(3)]
for s in shapes:
    print(type(s).__name__, 'area =', s.area())      # Polymorphism again!

---

## Part 5: Abstract Classes Can Have Normal Methods Too

An abstract class is not only a list of rules. It can also contain:
- an `__init__` for **shared data**
- **normal methods** that all children get for free (inheritance!)

| Inside an abstract class | Children must write it? |
|--------------------------|-------------------------|
| `@abstractmethod` method | ✅ **Yes**, every child must |
| Normal method | ❌ No, children just inherit it |
| `__init__` | ❌ No, children call it with `super().__init__()` |

In this example, `describe()` is written **once** in the parent, and it uses `area()` and `perimeter()` which each child writes in its own way:

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    def __init__(self, name):                # Normal __init__ (shared)
        self.name = name

    @abstractmethod
    def area(self):                          # Children MUST write
        pass

    @abstractmethod
    def perimeter(self):                     # Children MUST write
        pass

    def describe(self):                      # Normal method - children get it free
        return f'{self.name}: area = {self.area()}, perimeter = {self.perimeter()}'


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__('Rectangle')
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Circle(Shape):
    def __init__(self, radius):
        super().__init__('Circle')
        self.radius = radius

    def area(self):
        return round(3.14159 * self.radius ** 2, 2)

    def perimeter(self):
        return round(2 * 3.14159 * self.radius, 2)


for shape in [Rectangle(4, 5), Circle(3)]:
    print(shape.describe())                  # describe() was written only ONCE!

### ✏️ Quick Check

Will this code work? Guess, then click.

```python
class Triangle(Shape):
    def __init__(self, a, b, c):
        super().__init__('Triangle')
        self.a, self.b, self.c = a, b, c

    def perimeter(self):
        return self.a + self.b + self.c

t = Triangle(3, 4, 5)
```

<details>
<summary><b>Click to see answer</b></summary>

❌ **No!** `Triangle` wrote `perimeter()` but **forgot `area()`**. Since `area()` is an `@abstractmethod`, Python raises:

`TypeError: Can't instantiate abstract class Triangle ...`

Every abstract method must be written — not just some of them.

</details>

---

## Part 6: Real Example — A Payment System

An online shop accepts **Card**, **UPI** and **Net Banking**. Every payment method **must** be able to `pay()` and `refund()`.

```
            ┌──────────────────────────────┐
            │   PaymentMethod  (ABSTRACT)  │  ← the contract
            │   pay(amount)     must write │
            │   refund(amount)  must write │
            └──────────────┬───────────────┘
            ┌──────────────┼───────────────┐
            ▼              ▼               ▼
      ┌──────────┐   ┌──────────┐   ┌────────────┐
      │   Card   │   │   UPI    │   │ NetBanking │  ← real classes,
      │ pay()  ✅│   │ pay()  ✅│   │ pay()    ✅│    each does it
      │refund()✅│   │refund()✅│   │ refund() ✅│    in its own way
      └──────────┘   └──────────┘   └────────────┘
```

The `checkout()` function only knows about **`PaymentMethod`**. It doesn't care *how* each one works — only that it **has** a `pay()` method. That's abstraction!

In [ ]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):                    # THE CONTRACT

    @abstractmethod
    def pay(self, amount):
        pass

    @abstractmethod
    def refund(self, amount):
        pass


class Card(PaymentMethod):
    def __init__(self, card_number):
        self.last4 = card_number[-4:]

    def pay(self, amount):
        # In a real app: talk to the bank, check the CVV, check for fraud...
        return f'💳 Charged Rs.{amount} to card ending {self.last4}'

    def refund(self, amount):
        return f'💳 Refunded Rs.{amount} to card ending {self.last4}'


class UPI(PaymentMethod):
    def __init__(self, upi_id):
        self.upi_id = upi_id

    def pay(self, amount):
        # In a real app: open the UPI app, wait for confirmation...
        return f'📱 Paid Rs.{amount} via UPI ({self.upi_id})'

    def refund(self, amount):
        return f'📱 Refunded Rs.{amount} to {self.upi_id}'


class NetBanking(PaymentMethod):
    def __init__(self, bank):
        self.bank = bank

    def pay(self, amount):
        return f'🏦 Paid Rs.{amount} via {self.bank} Net Banking'

    def refund(self, amount):
        return f'🏦 Refunded Rs.{amount} to your {self.bank} account'


def checkout(method, amount):                # Works with ANY PaymentMethod
    print(method.pay(amount))


checkout(Card('1234-5678-9012-3456'), 4999)
checkout(UPI('rahul@okhdfc'), 299)
checkout(NetBanking('HDFC'), 15000)

✅ Want to add **Wallet** payments next month? Just write `class Wallet(PaymentMethod)` with `pay()` and `refund()`. **`checkout()` doesn't change at all.** And if you forget `refund()`, Python tells you **immediately**.

---

## Part 7: Abstraction vs Encapsulation

Students often mix these two up. Both "hide" something, but they hide **different** things:

| | **Abstraction** | **Encapsulation** (Day 17) |
|---|---|---|
| **Hides** | the **complex work** (the *how*) | the **data** |
| **Goal** | make it **simple** to use | make it **safe** to use |
| **Question** | *"What can it do?"* | *"Who can touch the data?"* |
| **Python tools** | `ABC`, `@abstractmethod`, methods | `_protected`, `__private`, `@property` |
| **ATM example 🏧** | Screen shows just "Withdraw" — you don't see the network and database work | The cash is locked in a vault — you can't grab it directly |

They usually work **together**. In this small class, find both:

In [ ]:
class BankAccount:
    def __init__(self, holder):
        self.holder = holder
        self.__balance = 0                   # ENCAPSULATION: data is hidden

    def deposit(self, amount):               # ABSTRACTION: one simple method...
        if amount <= 0:                      # ...hides the checking work inside
            print('Amount must be positive')
            return
        self.__balance += amount
        print(f'Deposited Rs.{amount}')

    @property
    def balance(self):                       # ENCAPSULATION: read-only access
        return self.__balance


acc = BankAccount('Amit')
acc.deposit(1000)
acc.deposit(-50)
print('Balance:', acc.balance)

> **One-line rule:**
> **Abstraction** hides the **complexity**. **Encapsulation** hides the **data**.

---

## Part 8: Mini Project — Library System

Let's use **everything** from Days 16 to 20 in one small project.

| Concept | Where it's used |
|---------|-----------------|
| **Abstraction** | `LibraryItem` is abstract — every item **must** write `get_type()` and `get_creator()` |
| **Inheritance** | `Book`, `DVD`, `Magazine` inherit from `LibraryItem` |
| **Encapsulation** | `__available` and `__items` are private; `title` is a read-only property |
| **Polymorphism** | `Library` calls `item.get_info()` without caring which kind of item it is |
| **Magic method** (Day 18) | `__len__` lets us write `len(library)` |

```
                   LibraryItem (ABSTRACT)
                   - title, __available
                   - get_type()     ← must write
                   - get_creator()  ← must write
                   - checkout(), return_item(), get_info()  ← shared
                            │
            ┌───────────────┼───────────────┐
            ▼               ▼               ▼
          Book             DVD           Magazine

   Library  ──HAS──►  a list of LibraryItems
```

### Step 1 — The abstract base class

In [ ]:
from abc import ABC, abstractmethod


class LibraryItem(ABC):
    def __init__(self, title):
        self.__title = title
        self.__available = True              # ENCAPSULATION: private

    @property
    def title(self):                         # read-only
        return self.__title

    @property
    def is_available(self):                  # read-only
        return self.__available

    # ----- The contract: every item MUST write these -----
    @abstractmethod
    def get_type(self):
        pass

    @abstractmethod
    def get_creator(self):
        pass

    # ----- Shared methods: every item gets these for free -----
    def checkout(self):
        if not self.__available:
            return f"❌ '{self.title}' is already checked out"
        self.__available = False
        return f"✅ Checked out: '{self.title}'"

    def return_item(self):
        if self.__available:
            return f"ℹ️ '{self.title}' is already in the library"
        self.__available = True
        return f"📥 Returned: '{self.title}'"

    def get_info(self):
        status = 'IN ' if self.__available else 'OUT'
        return f'[{status}] {self.get_type():9} | {self.title} - {self.get_creator()}'

### Step 2 — The real item types (each fulfils the contract)

In [ ]:
class Book(LibraryItem):
    def __init__(self, title, author):
        super().__init__(title)
        self.author = author

    def get_type(self):
        return 'Book'

    def get_creator(self):
        return f'by {self.author}'


class DVD(LibraryItem):
    def __init__(self, title, director):
        super().__init__(title)
        self.director = director

    def get_type(self):
        return 'DVD'

    def get_creator(self):
        return f'directed by {self.director}'


class Magazine(LibraryItem):
    def __init__(self, title, issue):
        super().__init__(title)
        self.issue = issue

    def get_type(self):
        return 'Magazine'

    def get_creator(self):
        return f'issue #{self.issue}'

### Step 3 — The `Library` that manages all items

Notice: `Library` never mentions `Book`, `DVD` or `Magazine`. It only knows about **`LibraryItem`**. Add a new item type tomorrow (like `AudioBook`) and this class **doesn't change**.

In [ ]:
class Library:
    def __init__(self, name):
        self.name = name
        self.__items = []                    # ENCAPSULATION: private list

    def add(self, item):
        if not isinstance(item, LibraryItem):
            raise TypeError('Only LibraryItem objects can be added')
        self.__items.append(item)

    def find(self, title):
        for item in self.__items:
            if item.title.lower() == title.lower():
                return item
        return None

    def show_all(self):
        print(f'--- {self.name} ---')
        for item in self.__items:
            print(' ', item.get_info())      # POLYMORPHISM

    def __len__(self):                       # MAGIC METHOD (Day 18)
        return len(self.__items)

### Step 4 — Run the library!

In [ ]:
library = Library('City Library')
library.add(Book('The Alchemist', 'Paulo Coelho'))
library.add(Book('Wings of Fire', 'A.P.J. Abdul Kalam'))
library.add(DVD('Inception', 'Christopher Nolan'))
library.add(Magazine('National Geographic', 245))

print('Total items:', len(library))
library.show_all()

print()
print(library.find('The Alchemist').checkout())
print(library.find('Inception').checkout())
print(library.find('The Alchemist').checkout())       # already out
print(library.find('The Alchemist').return_item())

print()
library.show_all()

In [ ]:
# Try to break it!
try:
    item = LibraryItem('Mystery')                     # abstract -> blocked
except TypeError as e:
    print('Blocked:', e)

try:
    library.add('Just a string')                      # not a LibraryItem -> blocked
except TypeError as e:
    print('Blocked:', e)

try:
    library.find('Inception').title = 'Hacked'        # read-only property -> blocked
except AttributeError as e:
    print('Blocked:', e)

---

## Part 9: Common Beginner Mistakes

### ❌ Mistake 1: Forgetting to inherit from `ABC`

This is the sneakiest mistake. Without `(ABC)`, the `@abstractmethod` decorator **does nothing** — and you get no error at all!

In [ ]:
from abc import abstractmethod


class Animal:                            # ❌ forgot (ABC)
    @abstractmethod
    def speak(self):
        pass


a = Animal()                             # No error! The contract is NOT enforced
print('Created an Animal - but we did not want that!')
print(a.speak())                         # None

# ✅ Fix:  class Animal(ABC):

### ❌ Mistake 2: Spelling the method name differently in the child

For Python, `get_area` and `area` are **two different methods**. The child didn't write `area()`, so the contract is broken.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        pass


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def get_area(self):                  # ❌ Wrong name! Should be area()
        return self.side ** 2


try:
    Square(4)
except TypeError as e:
    print('Error:', e)

### ❌ Mistake 3: Thinking abstraction means "put `pass` everywhere"

Abstract classes are **not** only empty methods. Put **shared code** in the parent as normal methods (like `describe()` in Part 5, or `checkout()` in Part 8). Only the parts that are **different for each child** should be `@abstractmethod`.

---

## Part 10: Practice Exercises

Try each exercise in the empty cell **before** looking at the solutions at the end.

---

### Exercise 1: Animals ⭐
- Create an abstract class `Animal` with an abstract method `sound()`
- Create `Dog`, `Cat` and `Cow` that each return their own sound
- Check that `Animal()` gives a `TypeError`

```python
for a in [Dog(), Cat(), Cow()]:
    print(a.sound())       # Woof!  Meow!  Moo!
```

In [ ]:
# Write your Exercise 1 solution here

### Exercise 2: Notifications ⭐⭐
- Abstract class `Notification` with an abstract method `send(user, message)`
- `EmailNotification`, `SMSNotification`, `WhatsAppNotification` — each returns a different text
- Write a function `broadcast(user, message)` that sends the message through **all three**

```python
broadcast('Amit', 'Your order is ready')
# 📧 Email to Amit: Your order is ready
# 📱 SMS to Amit: Your order is ready
# 💬 WhatsApp to Amit: Your order is ready
```

In [ ]:
# Write your Exercise 2 solution here

### Exercise 3: Vehicle Rental ⭐⭐⭐
- Abstract class `Vehicle(name, daily_rate)` with:
  - an abstract method `rental_cost(days)`
  - a **normal** method `bill(days)` that returns `'<name> for <days> days: Rs.<cost>'`
- `Car`: cost = `daily_rate × days`
- `Bike`: cost = `daily_rate × days`, but **20% off** if `days >= 7`

```python
Car('Swift', 2000).bill(3)     # 'Swift for 3 days: Rs.6000'
Bike('Activa', 500).bill(7)    # 'Activa for 7 days: Rs.2800.0'
```

In [ ]:
# Write your Exercise 3 solution here

---

### ✅ Solutions

*(Try the exercises first! Then run these to compare.)*

In [ ]:
# Solution 1: Animals
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass


class Dog(Animal):
    def sound(self):
        return 'Woof!'


class Cat(Animal):
    def sound(self):
        return 'Meow!'


class Cow(Animal):
    def sound(self):
        return 'Moo!'


for a in [Dog(), Cat(), Cow()]:
    print(type(a).__name__, '->', a.sound())

try:
    Animal()
except TypeError as e:
    print('Blocked:', e)

In [ ]:
# Solution 2: Notifications
from abc import ABC, abstractmethod


class Notification(ABC):
    @abstractmethod
    def send(self, user, message):
        pass


class EmailNotification(Notification):
    def send(self, user, message):
        return f'📧 Email to {user}: {message}'


class SMSNotification(Notification):
    def send(self, user, message):
        return f'📱 SMS to {user}: {message}'


class WhatsAppNotification(Notification):
    def send(self, user, message):
        return f'💬 WhatsApp to {user}: {message}'


def broadcast(user, message):
    channels = [EmailNotification(), SMSNotification(), WhatsAppNotification()]
    for channel in channels:
        print(channel.send(user, message))


broadcast('Amit', 'Your order is ready')

In [ ]:
# Solution 3: Vehicle Rental
from abc import ABC, abstractmethod


class Vehicle(ABC):
    def __init__(self, name, daily_rate):
        self.name = name
        self.daily_rate = daily_rate

    @abstractmethod
    def rental_cost(self, days):
        pass

    def bill(self, days):                    # normal (shared) method
        return f'{self.name} for {days} days: Rs.{self.rental_cost(days)}'


class Car(Vehicle):
    def rental_cost(self, days):
        return self.daily_rate * days


class Bike(Vehicle):
    def rental_cost(self, days):
        cost = self.daily_rate * days
        if days >= 7:
            cost = cost * 0.8                # 20% off for a week or more
        return cost


print(Car('Swift', 2000).bill(3))
print(Bike('Activa', 500).bill(3))
print(Bike('Activa', 500).bill(7))

---

## Part 11: Cheat Sheet — What We Learned Today

| Concept | Syntax | Meaning |
|---------|--------|---------|
| **Abstraction** | — | Hide the complex *how*, show a simple *what* |
| **Import** | `from abc import ABC, abstractmethod` | Get the tools |
| **Abstract class** | `class Shape(ABC):` | A contract — cannot create objects from it |
| **Abstract method** | `@abstractmethod` above a method | Every child **must** write this method |
| **Concrete (real) class** | `class Square(Shape):` + all methods written | Can create objects |
| **Missing a method?** | — | `TypeError` as soon as you create the object (**fail fast**) |

### Template to remember
```python
from abc import ABC, abstractmethod

class Parent(ABC):
    @abstractmethod
    def must_do(self):
        pass

class Child(Parent):
    def must_do(self):
        return 'Done!'
```

---

### 🏛️ The 4 Pillars of OOP — Complete!

| Pillar | Day | One-line meaning | Python tools |
|--------|-----|------------------|--------------|
| 🔒 **Encapsulation** | Day 17 | Hide the **data**, allow access only through safe methods | `_x`, `__x`, `@property` |
| 🧬 **Inheritance** | Day 19 | A child class **reuses** everything from its parent | `class Child(Parent)`, `super()` |
| 🎭 **Polymorphism** | Day 19 | **Same method call**, different behaviour per object | method overriding |
| 🎨 **Abstraction** | Day 20 | Hide the **complexity**, force a common contract | `ABC`, `@abstractmethod` |

> 🎉 **You now know all four pillars of Object-Oriented Programming!**

---

### 🔮 Next: Day 21 — OOP Mini Project
We'll use all four pillars together to build a complete project.